---

Image datasets and measurement

---

In [7]:
# autoload
%load_ext autoreload
%autoreload 2

# Load PGL libraries and start a PGL window
from pgl import pgl
from pgl.pglImage import pglImageDatabase, pglImageDatabaseWithManifest, pglImage
from pgl.pglMessages import pglMessages
from pgl.pglExperiment import pglTask, pglExperiment
from pgl.pglParameter import pglParameter, pglParameterBatch
import numpy as np

pgl = pgl()

# close any existing windows
pgl.cleanUp()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(pglBase) Main library closed
================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
(pgl:checkOS) Running on Mac mini (Mac14,3) with macOS version: 15.6.1
(pgl:checkOS) Apple M2 Cores: 8 (4 performance and 4 efficiency) Memory: 16 GB
(pgl:checkOS) GPU: Apple M2 (Built-In) 10 cores, Metal 3 support
(pgl:checkOS)   HP ZR2440w [Main Display]: 1920 x 1200 (WUXGA - Widescreen Ultra eXtended Graphics Array) (Unknown type) GammaTable size: 1024
(pgl:

---

Make a task to display images

---

In [ ]:
class pglImageTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Image Task"
        self.settings.nTrials = 100
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            #'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000",
            #'manifestColumnNames': {'filenameColumn':"filename",'indexColumn':"index",'captionColumn':"caption_1"},
            'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",
            'manifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"test_image_nr",'captionColumn':"concept"},
            'imdb': None,
            'nImages': 10,
            'imdbCatch': None,
            'catchImagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_catch_img_12reps",
            'catchManifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"catch_nr",'captionColumn':"original_filename"},
            'nCatchImages': 3,
            'imageSize': 18,
            'nImagesPerTrial': 10,
            'catchTrialEvery': 12,
            'catchBatchEvery': None,
            'numConditionBits': 10
        }        
        p = self.settings.fixedParameters
        
        # setup digitalIO
        #from pgl import pglLabJack
        #self.digIO = pglLabJack()
        from pgl import pglDataPixx
        self.dataPixx = pglDataPixx()

        if self.dataPixx.isActive:
            try:
                # initialize digital ports
                self.dataPixx.setupConditions(numBits=p['numConditionBits'])
                # keep a pointer for digIO (so in principle the code can work with a differnt digIO device)
                self.digIO = self.dataPixx
                # add to pgl, so that pgl.poll() will return button press events
                self.pgl.devicesAdd(self.dataPixx)
            except Exception as e:
                pglMessages.warning(f"Error setting up dataPixx: {e}")    
                self.digIO = None
                self.dataPixx = None        
        else:
            pglMessages.warning("DataPixx not active, digitalIO and button presses will not function")
            self.digIO = None
            self.dataPixx = None


        # set seglens, 
        # 1st segment is image display
        # 2nd segment is blank
        self.settings.seglen = [0.5, 0.5] * p['nImagesPerTrial']

        # initialize image database using parameters set from fixedParameters
        imdb = pglImageDatabaseWithManifest(
            p['imagesDirectory'],
            filenameColumn=p['manifestColumnNames']['filenameColumn'],
            indexColumn=p['manifestColumnNames']['indexColumn'],
            captionColumn=p['manifestColumnNames']['captionColumn'],
        )
        if imdb.nStimuli==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        p['imdb'] = imdb
        
        # initialize catch image database using parameters set from fixedParameters
        imdbCatch = pglImageDatabaseWithManifest(
            p['catchImagesDirectory'],
            filenameColumn=p['catchManifestColumnNames']['filenameColumn'],
            indexColumn=p['catchManifestColumnNames']['indexColumn'],
            captionColumn=p['catchManifestColumnNames']['captionColumn'],
        )
        if imdbCatch.nStimuli==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        p['imdbCatch'] = imdbCatch
        
        # preload images
        for iImage in range(p['nImages']):
            imdb.preload(iImage)
        for iImage in range(p['nCatchImages']):
            imdbCatch.preload(iImage)
            
        # add parameter for image number
        imageNumParameter = pglParameterBatch('imageNum',np.arange(p['nImages']),batchSize=p['nImagesPerTrial'], catchTrialEvery=p['catchTrialEvery'], catchBatchEvery=p['catchBatchEvery'])
        self.addParameter(imageNumParameter)
        
        # create a self-tracked parameter of the catch images
        self.state.catchNumParameter = pglParameter('catchNum',np.arange(p['nCatchImages']))
        
        # set current values
        self.state.currentImage = None
        self.state.isCatch= False
        self.state.wasCatch = False

    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment % 2 == 0: 
            # remember if the last segment was a catch
            self.state.wasCatch = self.state.isCatch
            self.state.gotResponse = False
            # set fixation colro
            self.state.fixColor = 1
            # get the current image number
            imageNums = self.currentParams['imageNum']
            if imageNums:
                # load an image
                imageNum = imageNums[int(self.state.currentSegment/2)]
                if imageNum:
                    # get the image data
                    img = self.settings.fixedParameters['imdb'].get(imageNum)
                    self.state.isCatch = False
                else:
                    # get a catch image
                    catchNum = self.state.catchNumParameter.get()['catchNum']
                    img = self.settings.fixedParameters['imdbCatch'].get(catchNum)    
                    self.state.isCatch = True
                    imageNums[int(self.state.currentSegment/2)] = -catchNum
                img.convert("RGB")
                print(f"img: {img}")
                # turn into a pglImage
                self.state.currentImage = self.pgl.imageCreate(np.array(img))
                # send digital pulse
                #if imageNum and self.digIO is not None: self.digIO.writeCondition(imageNum)
            else:
                # catch trial
                self.state.currentImage = None

    ########################
    # handleSubjectResponse
    ########################    
    def handleSubjectResponse(self, response, updateTime):
        '''
        Handle the subject response. Response will come in as an integer
        value of what button was pressed. The order of buttons is set
        in pgl.settings() in the field "responseKeys"
        '''
        # already received a response
        if self.state.gotResponse: return None
        # mark that we got a response
        self.state.gotResponse = True
        
        # if this was a catchTrial or the last one was, then it is correct
        if self.state.isCatch or self.state.wasCatch:
            correct = True 
            self.state.fixColor = [0,1,0]
        else:
            correct = False
            self.state.fixColor = [1,0,0]
            
        # return response type
        return correct

    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment % 2 == 0: 
            if self.state.currentImage:
                self.state.currentImage.display(height=self.settings.fixedParameters['imageSize'])
        
        # Draw ABC fixation cross from Thaler, Schütz, Goodale & Gegenfurtner (2013) Vision Research 76:31-42
        pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
        pgl.rect(-0.3,0,width=0.6,height=0.15,color=self.state.fixColor,hAlign='left',vAlign='center')
        pgl.rect(0,-0.3,width=0.15,height=0.6,color=self.state.fixColor,hAlign='center',vAlign='top')
        pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

        


---

Setup experiment

---

In [21]:
pgl.cleanUp()
#e = pglExperiment(pgl,settingsName='Cinema',experimentName='imageTask')
e = pglExperiment(pgl,experimentName='imageTask')

imageTask = pglImageTask(pgl)
e.addTask(imageTask)

(pglBase:removeOrphanedSockets) No orphaned sockets found in /Users/justin/Library/Containers/gru.mglMetal/Data
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglDataPixx->pglDataPixxBase:__init__) pypixxlib is not installed. Please install it to use DataPixx.
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglDataPixx:isActive) No DP library found
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglImageTask:__init__) DataPixx not active, digitalIO and button presses will not function
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglDataPixx->pglDataPixxBase:getError) (pglDataPixx:getError) Could not get error state: 'NoneType' object has no attribute 'DPxGetError'
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglImageDatabaseWithMani

AttributeError: 'pglImageDatabaseWithManifest' object has no attribute 'preloadImage'

---

run experiment

---

In [ ]:
e.initScreen()
e.run()

---

Load the image database

---

In [8]:
# load the database of images. Will check in directory for image formats that PIL
# knows about and make a list. This does not load the images, or check to see if they are valid
#imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000")
imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",filenameColumn="image_filename",indexColumn="test_image_nr",captionColumn="concept")

(pglImageDatabaseWithManifest->pglStimulusDatabase:__init__) Found 200 image files in things_200_img_12reps
(pglImageDatabaseWithManifest:__init__) Sorted and set captions from manifest


---

Print and display images

---

In [10]:
# display a single image
#imdb.images[111].display()

# print image metadata one-by-one, this may take some time because it 
# has to open each file 
#imdb.print() 
imdb.print()

(pglImageDatabaseWithManifest->pglStimulusDatabase:print) 0: alligator_14n.jpg 1: altar_13s.jpg 2: ashtray_14n.jpg 3: axe_14n.jpg 4: bamboo_13s.jpg 5: banana_13s.jpg 6:
beachball_16s.jpg 7: bean_13s.jpg 8: beaver_13n.jpg 9: bed_20n.jpg 10: beer_14s.jpg 11: bench_16n.jpg 12: bike_14s.jpg
13: blind_19s.jpg 14: boa_13s.jpg 15: boat_14n.jpg 16: bobsled_14s.jpg 17: brace_14s.jpg 18: brownie_14s.jpg 19:
bulldozer_22n.jpg 20: butterfly_16n.jpg 21: candelabra_14s.jpg 22: cheese_14n.jpg 23: chest1_14s.jpg 24:
chipmunk_16n.jpg 25: clipboard_14s.jpg 26: coat_rack_13s.jpg 27: cookie_15s.jpg 28: cow_16n.jpg 29: crank_16s.jpg 30:
crayon_15s.jpg 31: cufflink_16s.jpg 32: donut_15s.jpg 33: dough_19s.jpg 34: dragonfly_13s.jpg 35: drain_16s.jpg 36:
drawer_13s.jpg 37: dress_13s.jpg 38: earring_16n.jpg 39: easel_15s.jpg 40: ferris_wheel_20s.jpg 41: footprint_15s.jpg
42: fudge_14s.jpg 43: graffiti_17s.jpg 44: grape_20s.jpg 45: grate_13s.jpg 46: guacamole_18s.jpg 47: headlamp_14s.jpg
48: helicopter_25s.jpg 4

---

Display image dataset in a dialog

---

In [12]:
pgl.traitsDialog(imdb)

(pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/alligator_14n.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/alligator_14n.jpg
(pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/cookie_15s.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/cookie_15s.jpg
(pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/dolly_13s.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/dolly_13s.jpg
(pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/horseshoe_13n.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Deskto

In [16]:
img=imdb.get(0)

(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/alligator_14n.jpg


---

Test dataPixx

---

In [ ]:
from pgl import pglDataPixx
dataPixx = pglDataPixx()
dataPixx.enableButtonSchedules('press8')

In [ ]:
dataPixx.poll()

In [ ]:
dataPixx.setupConditions(numBits=8)


In [ ]:
from pgl import pgl
pgl = pgl()
for iCondition in range(256):
    dataPixx.writeCondition(iCondition)
    pgl.waitSecs(0.1)

---

Image database test code

---

---

Other test code

---

In [ ]:
from pgl.pglPipeline import pglRun, pglChoose
r = pglRun.load()
#fullDataPath = pglChoose.getExperimentPath()
#print(fullDataPath)
if r: r.print()

In [ ]:
# import pglLabJack and create an instance
from pgl import pglLabJack
pglLabJack = pglLabJack()

In [ ]:
# initialize digital ports
numChannels = 4
channelGroup = "DIO"

# set up channels
for channelNum in range(numChannels):
    pglLabJack.setupDigitalOutput(channelNum,channelGroup=channelGroup)
pglLabJack.setupDigitalOutputWord(channels=list(range(numChannels)))

#for i in range(15):
    # write a word
#    pglLabJack.digitalOutputWord(i+1)
    # wait
#    pgl.waitSecs(0.1)





In [ ]:
from pgl.pglTimestamp import pglTimestamp
for i in range(15):
    pglLabJack.digitalOutputWord(15)
    pglTimestamp.waitSecs(0.1)
#pglLabJack.wordMaxValue

In [ ]:
pglLabJack.digitalOutputWord(1)

In [ ]:
pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

pgl.flush()